# Ejercicio 3 - Consultas directas sobre archivos Parquet

Exploracion inicial de los Parquet de taxis amarillos y verdes **sin importarlos a una tabla**: cada consulta usa `read_parquet(...)`, `glob(...)`, `parquet_file_metadata(...)` o `parquet_schema(...)` directamente sobre `data/raw/`.

Las consultas viven en `sql/ejercicio3/` (una por archivo, con su objetivo y fuente en el encabezado). Este notebook las ejecuta y muestra; la documentacion completa (SQL + resultado + tiempo) se genera con:

```bash
python scripts/run_sql.py ejercicio3   # -> docs/resultados/ejercicio3.md
```

Las decisiones tomadas a partir de cada resultado estan en `docs/ejercicio3_exploracion.md`.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "scripts"))   # notebooks/ -> scripts/
from lab import conectar, ejecutar_sql, consulta, RAIZ_REPO

import pandas as pd
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

def correr(archivo):
    """Ejecuta un .sql, muestra sus metadatos y tiempo, y devuelve el DataFrame."""
    meta, df, seg = ejecutar_sql(con, archivo)
    print(f"[{meta.get('id')}] {meta.get('titulo', '')}")
    for clave in ("pregunta", "objetivo", "fuente"):
        if clave in meta:
            print(f"  {clave}: {meta[clave]}")
    print(f"  tiempo: {seg:.3f} s | filas: {len(df)}")
    return df

con = conectar(vistas=False)   # sin vistas: todo se lee directo de los Parquet

## 3.1 Cantidad de archivos

In [2]:
correr("sql/ejercicio3/3_01_cantidad_archivos.sql")

[3.1] Cantidad de archivos disponibles por tipo y anio
  objetivo: Contar los archivos Parquet descargados y listar los meses que cubren, para confirmar que no hay huecos en la serie mensual.
  fuente: data/raw/*/*/*.parquet (solo nombres de archivo, no se leen datos)
  tiempo: 0.044 s | filas: 2


,tipo,anio,archivos,meses
0,green,2026,8,"01, 02, 03, 04, 05, 06, 07, 08"
1,yellow,2026,8,"01, 02, 03, 04, 05, 06, 07, 08"


## 3.2 Cantidad de registros

Primero con los **metadatos** del footer de cada Parquet (no lee datos), luego con un `count(*)` sobre los archivos. Ambos totales deben coincidir.

In [3]:
por_archivo = correr("sql/ejercicio3/3_02_registros_por_archivo.sql")
por_archivo

[3.2a] Registros por archivo usando solo los metadatos Parquet
  objetivo: Obtener la cantidad de filas de cada archivo leyendo unicamente el footer del Parquet (sin escanear los datos) y detectar meses con volumen anormalmente bajo o alto.
  fuente: data/raw/*/*/*.parquet via parquet_file_metadata()
  tiempo: 0.042 s | filas: 16


,tipo,periodo,registros,row_groups,created_by
0,green,2026-01,40272,1,parquet-cpp-arrow version 16.1.0
1,green,2026-02,37373,1,parquet-cpp-arrow version 16.1.0
2,green,2026-03,44208,1,parquet-cpp-arrow version 16.1.0
3,green,2026-04,44238,1,parquet-cpp-arrow version 16.1.0
4,green,2026-05,44921,1,parquet-cpp-arrow version 16.1.0
5,green,2026-06,44163,1,parquet-cpp-arrow version 21.0.0
6,green,2026-07,41252,1,parquet-cpp-arrow version 21.0.0
7,green,2026-08,40687,1,parquet-cpp-arrow version 21.0.0
8,yellow,2026-01,3724889,4,parquet-cpp-arrow version 16.1.0
9,yellow,2026-02,3399866,4,parquet-cpp-arrow version 16.1.0


In [4]:
totales = correr("sql/ejercicio3/3_03_registros_totales.sql")
print("Suma de metadatos por tipo:")
print(por_archivo.groupby("tipo")["registros"].sum())
totales

[3.2b] Registros totales por tipo de taxi (escaneo con count)
  objetivo: Contar el total de registros consultando los Parquet y comparar con la suma de metadatos de 3.2a (deben coincidir).
  fuente: data/raw/yellow/*/*.parquet y data/raw/green/*/*.parquet
  tiempo: 0.105 s | filas: 3
Suma de metadatos por tipo:
tipo
green       337114
yellow    29703355
Name: registros, dtype: int64


,tipo,registros
0,yellow,29703355
1,green,337114
2,total,30040469


## 3.3 Columnas presentes

In [5]:
correr("sql/ejercicio3/3_04_columnas_yellow.sql")[["column_name", "column_type", "null"]]

[3.3a] Columnas y tipos de datos - taxis amarillos
  objetivo: Identificar las columnas y el tipo de dato que DuckDB infiere al leer todos los archivos amarillos en conjunto.
  fuente: data/raw/yellow/*/*.parquet
  tiempo: 0.055 s | filas: 21


,column_name,column_type,null
0,VendorID,INTEGER,YES
1,tpep_pickup_datetime,TIMESTAMP,YES
2,tpep_dropoff_datetime,TIMESTAMP,YES
3,passenger_count,BIGINT,YES
4,trip_distance,DOUBLE,YES
5,RatecodeID,BIGINT,YES
6,store_and_fwd_flag,VARCHAR,YES
7,PULocationID,INTEGER,YES
8,DOLocationID,INTEGER,YES
9,payment_type,BIGINT,YES


In [6]:
correr("sql/ejercicio3/3_05_columnas_green.sql")[["column_name", "column_type", "null"]]

[3.3b] Columnas y tipos de datos - taxis verdes
  objetivo: Identificar las columnas y el tipo de dato de los archivos verdes.
  fuente: data/raw/green/*/*.parquet
  tiempo: 0.027 s | filas: 22


,column_name,column_type,null
0,VendorID,INTEGER,YES
1,lpep_pickup_datetime,TIMESTAMP,YES
2,lpep_dropoff_datetime,TIMESTAMP,YES
3,store_and_fwd_flag,VARCHAR,YES
4,RatecodeID,BIGINT,YES
5,PULocationID,INTEGER,YES
6,DOLocationID,INTEGER,YES
7,passenger_count,BIGINT,YES
8,trip_distance,DOUBLE,YES
9,fare_amount,DOUBLE,YES


In [7]:
correr("sql/ejercicio3/3_06_comparacion_columnas.sql")

[3.3c] Comparacion de columnas entre amarillos y verdes
  objetivo: Ver que columnas son comunes, cuales son exclusivas de un tipo y cuales tienen distinto nombre o tipo, para disenar un esquema unificado.
  fuente: data/raw/yellow/*/*.parquet y data/raw/green/*/*.parquet
  tiempo: 0.053 s | filas: 25


,columna,tipo_yellow,tipo_green,estado
0,DOLocationID,INTEGER,INTEGER,comun
1,PULocationID,INTEGER,INTEGER,comun
2,RatecodeID,BIGINT,BIGINT,comun
3,VendorID,INTEGER,INTEGER,comun
4,cbd_congestion_fee,DOUBLE,DOUBLE,comun
5,congestion_surcharge,DOUBLE,DOUBLE,comun
6,extra,DOUBLE,DOUBLE,comun
7,fare_amount,DOUBLE,DOUBLE,comun
8,improvement_surcharge,DOUBLE,DOUBLE,comun
9,mta_tax,DOUBLE,DOUBLE,comun


## 3.4 Tipos de datos por archivo

Las filas con `n_tipos > 1` o `en_archivos < archivos_totales` aparecen primero: indican cambios de esquema entre archivos.

In [8]:
tipos = correr("sql/ejercicio3/3_07_tipos_por_archivo.sql")
tipos[(tipos.n_tipos > 1) | (tipos.en_archivos < tipos.archivos_totales)]

[3.4] Consistencia de tipos fisicos entre archivos
  objetivo: Revisar, columna por columna, si todos los archivos usan el mismo tipo Parquet. Una columna con mas de un tipo o presente en menos archivos que el total indica un cambio de esquema entre meses/anios.
  fuente: data/raw/*/*/*.parquet via parquet_schema()
  tiempo: 0.050 s | filas: 43


,tipo,columna,tipos_parquet,n_tipos,en_archivos,archivos_totales
0,green,request_source,BYTE_ARRAY (UTF8),1,3,8
22,yellow,request_source,BYTE_ARRAY (UTF8),1,3,8


In [9]:
tipos

,tipo,columna,tipos_parquet,n_tipos,en_archivos,archivos_totales
0,green,request_source,BYTE_ARRAY (UTF8),1,3,8
1,green,DOLocationID,INT32,1,8,8
2,green,PULocationID,INT32,1,8,8
3,green,RatecodeID,INT64,1,8,8
4,green,VendorID,INT32,1,8,8
5,green,cbd_congestion_fee,DOUBLE,1,8,8
6,green,congestion_surcharge,DOUBLE,1,8,8
7,green,ehail_fee,DOUBLE,1,8,8
8,green,extra,DOUBLE,1,8,8
9,green,fare_amount,DOUBLE,1,8,8


## 3.5 Muestra de registros

In [10]:
correr("sql/ejercicio3/3_08_muestra_yellow.sql")

[3.5a] Muestra reproducible de 10 registros - taxis amarillos
  objetivo: Inspeccionar registros reales para entender formato y valores. Se ordena por un hash de varias columnas en lugar de usar USING SAMPLE: el resultado es pseudoaleatorio pero identico en cada ejecucion, sin importar cuantos hilos use DuckDB.
  fuente: data/raw/yellow/*/*.parquet
  tiempo: 2.534 s | filas: 10


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,request_source
0,2,2026-07-23 02:07:37,2026-07-23 02:37:35,<NA>,8.79,<NA>,NaN,48,61,0,33.04,0.0,0.5,0.00,0.0,1.0,37.79,NaN,NaN,0.75,A
1,2,2026-08-11 08:25:53,2026-08-11 08:38:23,1,1.94,1,N,263,162,1,13.50,0.0,0.5,3.65,0.0,1.0,21.90,2.5,0.0,0.75,NaN
2,2,2026-06-08 22:35:26,2026-06-08 22:40:56,1,1.40,1,N,151,239,1,8.60,1.0,0.5,2.72,0.0,1.0,16.32,2.5,0.0,0.00,NaN
3,2,2026-03-13 20:16:32,2026-03-13 20:32:58,1,4.45,1,N,13,79,1,21.90,1.0,0.5,6.91,0.0,1.0,34.56,2.5,0.0,0.75,NaN
4,2,2026-02-27 17:58:08,2026-02-27 18:27:12,1,6.78,1,N,151,231,1,33.80,2.5,0.5,8.21,0.0,1.0,49.26,2.5,0.0,0.75,NaN
5,2,2026-08-26 07:31:32,2026-08-26 07:37:09,1,0.85,1,N,170,161,1,7.20,0.0,0.5,2.39,0.0,1.0,14.34,2.5,0.0,0.75,NaN
6,2,2026-08-12 13:49:12,2026-08-12 14:08:18,<NA>,3.01,<NA>,NaN,237,24,0,19.94,0.0,0.5,0.00,0.0,1.0,23.94,NaN,NaN,0.00,HV0003
7,1,2026-03-13 17:48:01,2026-03-13 18:00:44,1,2.10,99,N,55,108,1,19.50,0.0,0.5,0.00,0.0,0.0,20.00,0.0,0.0,0.00,NaN
8,2,2026-03-02 13:01:15,2026-03-02 13:06:01,1,1.01,1,N,236,238,1,7.20,0.0,0.5,2.24,0.0,1.0,13.44,2.5,0.0,0.00,NaN
9,2,2026-04-09 17:33:06,2026-04-09 17:56:00,2,2.13,1,N,236,161,1,20.50,2.5,0.5,5.55,0.0,1.0,33.30,2.5,0.0,0.75,NaN


In [11]:
correr("sql/ejercicio3/3_09_muestra_green.sql")

[3.5b] Muestra reproducible de 10 registros - taxis verdes
  objetivo: Inspeccionar registros reales para entender formato y valores. Se ordena por un hash de varias columnas en lugar de usar USING SAMPLE: el resultado es pseudoaleatorio pero identico en cada ejecucion, sin importar cuantos hilos use DuckDB.
  fuente: data/raw/green/*/*.parquet
  tiempo: 0.130 s | filas: 10


,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee,request_source
0,2,2026-07-29 08:01:49,2026-07-29 08:35:15,N,1,243,163,1,8.11,39.4,0.00,0.5,5.00,0.00,NaN,1.0,49.40,1,1,2.75,0.75,None
1,2,2026-06-04 07:52:48,2026-06-04 08:02:40,N,1,75,239,1,1.78,12.1,0.00,0.5,1.50,0.00,NaN,1.0,17.85,1,1,2.75,0.00,None
2,2,2026-08-26 23:19:11,2026-08-26 23:41:57,N,1,93,141,5,11.33,45.7,1.00,0.5,11.58,6.94,NaN,1.0,69.47,1,1,2.75,0.00,None
3,2,2026-08-15 16:21:11,2026-08-15 16:31:19,N,1,43,237,1,2.12,12.8,0.00,0.5,1.95,0.00,NaN,1.0,19.00,1,1,2.75,0.00,None
4,2,2026-08-18 11:45:08,2026-08-18 11:53:40,N,1,74,41,1,0.86,9.3,0.00,0.5,0.10,0.00,NaN,1.0,10.90,1,1,0.00,0.00,None
5,2,2026-03-23 07:20:26,2026-03-23 07:26:49,N,1,122,130,1,1.21,8.6,0.00,0.5,0.00,0.00,NaN,1.0,10.10,2,1,0.00,0.00,None
6,6,2026-05-24 00:14:58,2026-05-24 00:23:31,NaN,<NA>,20,241,<NA>,1.57,3.0,0.00,0.5,0.00,0.00,NaN,0.3,16.00,<NA>,<NA>,NaN,0.00,None
7,2,2026-04-21 12:13:16,2026-04-21 12:22:26,N,1,43,238,1,1.61,10.7,0.00,0.5,2.99,0.00,NaN,1.0,17.94,1,1,2.75,0.00,None
8,2,2026-07-30 08:43:56,2026-07-30 08:53:17,N,1,74,75,1,1.13,10.0,0.00,0.5,5.00,0.00,NaN,1.0,16.50,1,1,0.00,0.00,None
9,1,2026-05-04 17:26:48,2026-05-04 17:33:57,N,1,75,263,1,1.10,8.6,5.25,1.5,1.00,0.00,NaN,1.0,16.35,1,1,2.75,0.00,None


## 3.6 Problemas de calidad de datos

`SUMMARIZE` da un perfil de cada columna (min, max, cuartiles, % de nulos). Luego se cuantifican reglas concretas.

In [12]:
resumen_y = correr("sql/ejercicio3/3_10_resumen_yellow.sql")
resumen_y[["column_name", "column_type", "min", "max", "avg", "q50", "approx_unique", "null_percentage"]]

[3.6a] Perfil estadistico de columnas - taxis amarillos (SUMMARIZE)
  objetivo: Obtener min, max, promedio, cuartiles, valores unicos aproximados y porcentaje de nulos de cada columna para detectar rangos imposibles y columnas con muchos nulos.
  fuente: data/raw/yellow/*/*.parquet
  tiempo: 10.776 s | filas: 21


,column_name,column_type,min,max,avg,q50,approx_unique,null_percentage
0,VendorID,INTEGER,1,7,1.885739809526567,2,4,0.00
1,tpep_pickup_datetime,TIMESTAMP,2001-01-01 09:23:58,2026-08-31 23:59:59,2026-04-30 15:29:29.519432,2026-04-30 20:52:17.497972,16867928,0.00
2,tpep_dropoff_datetime,TIMESTAMP,2001-01-01 16:09:38,2026-09-01 20:16:00,2026-04-30 15:47:08.186216,2026-04-30 06:07:10.90544,16587125,0.00
3,passenger_count,BIGINT,0,9,1.2494318488564,1,11,25.98
4,trip_distance,DOUBLE,0.0,328522.2,5.552940149688975,1.8561724477771093,7217,0.00
5,RatecodeID,BIGINT,1,99,4.527471444398553,1,7,25.98
6,store_and_fwd_flag,VARCHAR,N,Y,NaN,NaN,2,25.98
7,PULocationID,INTEGER,1,265,161.57793013617484,161,290,0.00
8,DOLocationID,INTEGER,1,265,161.05139342003622,162,298,0.00
9,payment_type,BIGINT,0,5,0.8621735154160195,1,6,0.00


In [13]:
resumen_g = correr("sql/ejercicio3/3_11_resumen_green.sql")
resumen_g[["column_name", "column_type", "min", "max", "avg", "q50", "approx_unique", "null_percentage"]]

[3.6b] Perfil estadistico de columnas - taxis verdes (SUMMARIZE)
  objetivo: Igual que 3.6a para taxis verdes.
  fuente: data/raw/green/*/*.parquet
  tiempo: 0.260 s | filas: 22


,column_name,column_type,min,max,avg,q50,approx_unique,null_percentage
0,VendorID,INTEGER,1,6,2.328351833504393,2,3,0.00
1,lpep_pickup_datetime,TIMESTAMP,2008-12-31 17:35:31,2026-08-31 23:58:28,2026-05-02 13:25:51.096427,2026-05-04 11:04:44.750132,332702,0.00
2,lpep_dropoff_datetime,TIMESTAMP,2008-12-31 23:16:26,2026-09-02 09:39:37,2026-05-02 13:46:38.833759,2026-05-03 11:30:43.337993,396286,0.00
3,store_and_fwd_flag,VARCHAR,N,Y,NaN,NaN,2,14.47
4,RatecodeID,BIGINT,1,99,1.2549672434183374,1,7,14.47
5,PULocationID,INTEGER,1,265,97.32761024460568,75,263,0.00
6,DOLocationID,INTEGER,1,265,142.8768458147689,140,266,0.00
7,passenger_count,BIGINT,0,9,1.3006461144694266,1,11,14.47
8,trip_distance,DOUBLE,0.0,179830.92,13.349507644298376,2.067546496950198,2472,0.00
9,fare_amount,DOUBLE,-500.0,1676.7,17.014101995170655,13.176411613120365,4808,0.00


In [14]:
reglas = correr("sql/ejercicio3/3_12_reglas_calidad.sql")
# porcentaje sobre el total de cada tipo
pct = reglas.set_index("tipo")
(pct.drop(columns="registros").div(pct["registros"], axis=0) * 100).round(3).T

[3.6c] Conteo de registros que violan reglas basicas de calidad
  objetivo: Cuantificar cada tipo de problema (fechas fuera del mes del archivo, duraciones y distancias imposibles, montos negativos, pasajeros 0, nulos) para decidir que filtros aplicar en el analisis.
  fuente: data/raw/yellow/*/*.parquet y data/raw/green/*/*.parquet
  tiempo: 1.538 s | filas: 2


tipo,yellow,green
pickup_fuera_del_mes,0.000,0.029
dropoff_antes_pickup,0.000,0.001
duracion_cero,1.251,0.068
duracion_mas_6h,0.025,0.327
distancia_cero,3.206,3.623
distancia_mas_100mi,0.004,0.021
tarifa_negativa,0.530,0.296
total_negativo,0.545,0.303
pasajeros_cero,0.308,1.343
pasajeros_nulos,25.979,14.468


In [15]:
correr("sql/ejercicio3/3_13_fechas_fuera_de_mes.sql")

[3.6d] Detalle de fechas de pickup fuera del mes del archivo
  objetivo: Ver a que fechas corresponden los registros que no caen en el mes del archivo (bordes de mes vs. fechas claramente erroneas como 2008/2009).
  fuente: data/raw/yellow/*/*.parquet y data/raw/green/*/*.parquet
  tiempo: 0.513 s | filas: 30


,tipo,archivo_periodo,pickup_periodo,registros,pickup_min,pickup_max
0,yellow,2026-07,2026-08,38,2026-08-01 00:02:37,2026-08-05 20:54:00
1,green,2026-01,2026-02,18,2026-02-01 00:15:28,2026-02-01 21:08:36
2,yellow,2026-03,2026-02,15,2026-02-28 23:38:17,2026-02-28 23:59:47
3,yellow,2026-06,2026-04,15,2026-04-16 18:12:39,2026-04-20 20:50:27
4,yellow,2026-08,2026-07,14,2026-07-31 22:08:35,2026-07-31 23:59:13
5,yellow,2026-02,2026-01,12,2026-01-31 23:31:23,2026-01-31 23:59:47
6,green,2026-06,2026-05,12,2026-05-26 19:47:06,2026-05-31 23:57:01
7,green,2026-08,2026-07,12,2026-07-25 21:42:42,2026-07-31 23:58:54
8,yellow,2026-05,2026-04,11,2026-04-30 23:47:18,2026-04-30 23:59:17
9,green,2026-05,2026-04,8,2026-04-24 22:00:18,2026-04-30 23:54:33


In [16]:
correr("sql/ejercicio3/3_14_codigos_categoricos.sql")

[3.6e] Valores de las columnas categoricas codificadas
  objetivo: Comparar los codigos presentes (VendorID, RatecodeID, payment_type, store_and_fwd_flag) contra el diccionario de datos de la TLC para detectar codigos no documentados y nulos.
  fuente: data/raw/yellow/*/*.parquet y data/raw/green/*/*.parquet
  tiempo: 0.935 s | filas: 40


,tipo,columna,valor,registros
0,yellow,RatecodeID,1,20102072
1,yellow,RatecodeID,2,694936
2,yellow,RatecodeID,3,90052
3,yellow,RatecodeID,4,67285
4,yellow,RatecodeID,5,262614
5,yellow,RatecodeID,6,15
6,yellow,RatecodeID,99,769693
7,yellow,RatecodeID,NaN,7716688
8,yellow,VendorID,1,5467071
9,yellow,VendorID,2,23809774


In [17]:
correr("sql/ejercicio3/3_15_consistencia_montos.sql")

[3.6f] Consistencia entre total_amount y la suma de sus componentes
  objetivo: Verificar si total_amount es igual a la suma de tarifa, recargos, impuestos, propina y peajes. Diferencias grandes indican registros inconsistentes o componentes que no estan en el archivo.
  fuente: data/raw/yellow/*/*.parquet y data/raw/green/*/*.parquet
  tiempo: 1.330 s | filas: 2


,tipo,registros,coincide,no_coincide,pct_no_coincide,mediana_diferencia
0,yellow,29703355,18807599,10895756,36.682,2.5
1,green,337114,270144,66970,19.866,12.2


In [18]:
correr("sql/ejercicio3/3_16_duplicados.sql")

[3.6g] Posibles registros duplicados
  objetivo: Contar viajes repetidos con el mismo proveedor, hora de pickup y dropoff, zonas y monto total (un mismo viaje cargado dos veces).
  fuente: data/raw/yellow/*/*.parquet y data/raw/green/*/*.parquet
  tiempo: 1.839 s | filas: 2


,tipo,registros,combinaciones_unicas,registros_duplicados
0,yellow,29703355.0,29670561,32794.0
1,green,337114.0,337114,0.0
